# ROADGUARD — YOLOv8 License Plate Detection Training Pipeline
### AI-Powered Vehicle Document Verification System using ANPR

This notebook trains a custom YOLOv8 model for Indian License Plate localization using transfer learning from `yolov8n.pt`.
- **Dataset**: 5,583 annotated images (Train: 3,960 | Val: 1,182 | Test: 441)
- **Target Metric**: mAP@0.5 > 92%, Inference Latency < 25ms

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Install Dependencies

In [ ]:
!pip install ultralytics opencv-python matplotlib

## 3. Verify Dataset Structure

In [ ]:
import os
from pathlib import Path

DATASET_PATH = Path("unique dataset")
print(f"Dataset exists: {DATASET_PATH.exists()}")
print(f"Train images: {len(list((DATASET_PATH / 'train' / 'images').glob('*.*')))}")
print(f"Val images:   {len(list((DATASET_PATH / 'valid' / 'images').glob('*.*')))}")
print(f"Test images:  {len(list((DATASET_PATH / 'test' / 'images').glob('*.*')))}")

## 4. Train YOLOv8 Detector

In [ ]:
from ultralytics import YOLO

# Load pre-trained nano base weights
model = YOLO("yolov8n.pt")

# Execute transfer learning
results = model.train(
    data="unique dataset/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    patience=10,
    device=0 if torch.cuda.is_available() else "cpu",
    project="ROADGUARD_Plate_Detector",
    name="yolov8n_custom_run",
    exist_ok=True
)

## 5. Evaluate on Unseen Test Dataset

In [ ]:
metrics = model.val(split="test")
print(f"=== ROADGUARD TEST EVALUATION RESULTS ===")
print(f"mAP@0.50:        {metrics.box.map50 * 100:.2f}%")
print(f"mAP@0.50:0.95:   {metrics.box.map * 100:.2f}%")
print(f"Precision (P):   {metrics.box.p[0] * 100:.2f}%")
print(f"Recall (R):      {metrics.box.r[0] * 100:.2f}%")

## 6. Export and Display Training Curves

In [ ]:
from IPython.display import Image, display

run_dir = Path("ROADGUARD_Plate_Detector/yolov8n_custom_run")
results_png = run_dir / "results.png"
if results_png.exists():
    print("Displaying Loss & Metric Convergence Curves:")
    display(Image(filename=str(results_png)))

conf_matrix = run_dir / "confusion_matrix.png"
if conf_matrix.exists():
    print("Displaying Confusion Matrix:")
    display(Image(filename=str(conf_matrix)))